# 09 · Memory: Conversations 🧠

Every `responses.create` starts with a blank mind — unless we tell OpenAI which chat to continue.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-6-luna"
print("Ready!")

In [ ]:
bot = "You are SG Data Bot, a helpful assistant for life in Singapore. Keep replies short."

## 1. Without memory 🐟

In [ ]:
client.responses.create(model=MODEL, instructions=bot, input="Hi, I'm Ahmad. I live in Bedok and I cycle to work.")

response = client.responses.create(model=MODEL, instructions=bot, input="Where do I live?")
print(response.output_text)

The bot forgot already!

## 2. Memory way #1: `previous_response_id`
You used this in notebook 06. It says *"continue from this response"*.

In [ ]:
first = client.responses.create(model=MODEL, instructions=bot, input="Hi, I'm Ahmad. I live in Bedok and I cycle to work.")

second = client.responses.create(model=MODEL, instructions=bot, previous_response_id=first.id, input="Where do I live?")
print(second.output_text)

It works, but we must keep track of the **latest** response ID after every message. If the bot restarts, the ID is lost.

## 3. Memory way #2: a conversation 💾
A **conversation** is a chat history that OpenAI keeps for us. It has **one ID that never changes** — easy to save!

In [ ]:
conversation = client.conversations.create()

client.responses.create(model=MODEL, instructions=bot, conversation=conversation.id,
                        input="Hi, I'm Ahmad. I live in Bedok and I cycle to work.")

response = client.responses.create(model=MODEL, instructions=bot, conversation=conversation.id,
                                   input="Where do I live?")
print(response.output_text)

## 4. Peek inside the conversation

In [ ]:
items = client.conversations.items.list(conversation.id, order="asc")

for item in items.data:
    if item.type == "message":
        print(f"{item.role:>9}: {item.content[0].text}")

> 🧠 vs 📚 — **Memory** is what was *said* in the chat. The **vector store** (notebook 08) is the *documents* the bot can look things up in. Your bot uses both.

## 5. Make your bot's memory

In [ ]:
bot_conversation = client.conversations.create()

with open(".env", "a") as f:
    f.write(f"\nOPENAI_CONVERSATION_ID={bot_conversation.id}\n")

print("Saved! Open .env to check.")

> 👤 We're keeping it simple: **one bot, one person**. Everyone who chats with your bot shares this one conversation.

Your `.env` now has all three IDs your bot needs: `OPENAI_HDB_FILE_ID`, `OPENAI_VECTOR_STORE_ID` and `OPENAI_CONVERSATION_ID`.

## ✏️ Exercises
1. Count how many messages are in `conversation`.
2. Restart the kernel (⟳), run the setup cell, then chat using `conversation=os.environ["OPENAI_CONVERSATION_ID"]`. Tell it your name, restart again, and ask *"What's my name?"*

In [ ]:
# your code here